# 09 — Cross-encoder Reranking
**Project:** Semantic Book Recommender — IT4142 HUST  
**Owner:** M2d  
**Input:**
- `data/processed/books_with_emotions.csv`
- `data/chroma_db/` (BGE-small embeddings)
- `data/eval/test_queries.json`

**Output:**
- `reports/evaluation_final.json` — updated với Reranking scores
- `reports/figures/eval_reranking_analysis.png`
- `reports/figures/eval_latency_quality.png` — scatter: latency vs P@5 cho cả 5 model

---
## Bi-encoder vs Cross-encoder

```
Bi-encoder (BGE-small)          Cross-encoder (ms-marco-MiniLM)
─────────────────────           ────────────────────────────────
Query → embedding               (Query, Document) → relevance score
Doc   → embedding
score = dot product             score = full attention across both

✓ Fast (pre-compute doc embeds) ✓ More accurate
✗ Less accurate                 ✗ Cannot pre-compute (query-dependent)
                                ✗ Too slow for full corpus
```

**Solution: 2-stage pipeline**
```
Stage 1 — Recall  : BGE-small → top-20 candidates    (~50ms)
Stage 2 — Precision: Cross-encoder → rerank → top-5  (~150ms)
                                                       ──────
                                                       ~200ms total
```

## 0. Setup

In [ ]:
# pip install sentence-transformers chromadb
import pandas as pd
import numpy as np
import pickle, json, time
import scipy.sparse as sp
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
import chromadb
from chromadb.config import Settings

DATA_PATH   = Path('data/processed/books_with_emotions.csv')
MODEL_PATH  = Path('models')
CHROMA_PATH = Path('data/chroma_db')
EVAL_PATH   = Path('data/eval/test_queries.json')
REPORT_PATH = Path('reports')
FIGURE_PATH = Path('reports/figures')
FIGURE_PATH.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 150,
    'figure.facecolor': 'white', 'axes.facecolor': '#F9F9F9',
    'axes.spines.top': False, 'axes.spines.right': False,
})
MODEL_COLORS = {
    'TF-IDF'   : '#6B8CBA',
    'BM25'     : '#F4A261',
    'Semantic' : '#2A9D8F',
    'Hybrid'   : '#E76F51',
    'Reranking': '#8338EC',
}
print('Setup OK')

## 1. Load Data & Models

In [ ]:
df = pd.read_csv(DATA_PATH)
df['isbn13'] = df['isbn13'].astype(str)
print(f'Books: {len(df):,}')

with open(EVAL_PATH) as f:
    test_queries = json.load(f)
print(f'Test queries: {len(test_queries)}')

In [ ]:
# --- Stage 1: BGE-small bi-encoder ---
print('Loading BGE-small...')
bi_encoder = SentenceTransformer('BAAI/bge-small-en-v1.5')
BGE_PREFIX = 'Represent this sentence for searching relevant passages: '

chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_PATH),
    settings=Settings(anonymized_telemetry=False),
)
collection = chroma_client.get_collection('books')
print(f'ChromaDB count: {collection.count():,}')

# --- Stage 2: Cross-encoder ---
# ms-marco-MiniLM-L-6-v2: lightweight, trained on MS MARCO passage ranking
# Alternatives (heavier but more accurate):
#   'cross-encoder/ms-marco-MiniLM-L-12-v2'
#   'cross-encoder/ms-marco-electra-base'
CROSS_ENCODER_MODEL = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
print(f'Loading cross-encoder: {CROSS_ENCODER_MODEL}')
t0 = time.time()
cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL, max_length=512)
print(f'Cross-encoder loaded in {time.time()-t0:.1f}s')

# --- Baselines for comparison ---
print('Loading TF-IDF + BM25...')
with open(MODEL_PATH / 'tfidf_vectorizer.pkl', 'rb') as f:
    vectorizer = pickle.load(f)
tfidf_matrix = sp.load_npz(MODEL_PATH / 'tfidf_matrix.npz')
with open(MODEL_PATH / 'bm25_index.pkl', 'rb') as f:
    bm25 = pickle.load(f)

print('All models loaded ✓')

## 2. Search Functions

In [ ]:
def search_tfidf(query, top_k=10):
    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, tfidf_matrix).flatten()
    idx = np.argsort(scores)[::-1][:top_k]
    res = df.iloc[idx][['isbn13', 'title', 'authors', 'categories', 'description', 'average_rating']].copy()
    res['score'] = scores[idx]
    return res.reset_index(drop=True)

def search_bm25(query, top_k=10):
    scores = bm25.get_scores(query.lower().split())
    idx = np.argsort(scores)[::-1][:top_k]
    res = df.iloc[idx][['isbn13', 'title', 'authors', 'categories', 'description', 'average_rating']].copy()
    res['score'] = scores[idx]
    return res.reset_index(drop=True)

def search_semantic(query, top_k=10):
    q_emb = bi_encoder.encode([BGE_PREFIX + query], normalize_embeddings=True)
    results = collection.query(
        query_embeddings=q_emb.tolist(),
        n_results=top_k,
        include=['metadatas', 'distances', 'documents'],
    )
    rows = [{
        'isbn13'     : results['ids'][0][i],
        'title'      : results['metadatas'][0][i].get('title', ''),
        'authors'    : results['metadatas'][0][i].get('authors', ''),
        'categories' : results['metadatas'][0][i].get('categories', ''),
        'description': results['documents'][0][i],
        'score'      : round(1 - results['distances'][0][i], 4),
    } for i in range(len(results['ids'][0]))]
    return pd.DataFrame(rows)

print('Base search functions ready')

## 3. Cross-encoder Reranking Pipeline

In [ ]:
def rerank_with_cross_encoder(
    query: str,
    candidates: pd.DataFrame,
    top_k: int = 5,
) -> pd.DataFrame:
    """
    Rerank candidate documents using a cross-encoder.

    Args:
        query      : original search query
        candidates : DataFrame with 'description' column
        top_k      : number of results to return after reranking

    Returns:
        Top-k reranked DataFrame with 'rerank_score' column
    """
    if candidates.empty:
        return candidates

    # Build (query, passage) pairs
    pairs = [(query, desc) for desc in candidates['description']]

    # Cross-encoder scores — higher = more relevant
    scores = cross_encoder.predict(pairs, show_progress_bar=False)

    result = candidates.copy()
    result['rerank_score'] = scores
    result = result.sort_values('rerank_score', ascending=False).head(top_k)
    return result.reset_index(drop=True)


def search_with_reranking(
    query: str,
    top_k: int = 5,
    candidate_pool: int = 20,
    first_stage: str = 'semantic',  # 'semantic' | 'hybrid'
) -> pd.DataFrame:
    """
    Full 2-stage pipeline:
      Stage 1: bi-encoder retrieval → candidate pool
      Stage 2: cross-encoder reranking → top-k final

    Args:
        query          : natural language query
        top_k          : final number of results
        candidate_pool : how many candidates to retrieve in stage 1
        first_stage    : which model to use for stage 1
    """
    # Stage 1 — fast retrieval
    if first_stage == 'semantic':
        candidates = search_semantic(query, top_k=candidate_pool)
    elif first_stage == 'hybrid':
        # Import from notebook 08 logic inline
        from collections import defaultdict
        bm25_scores = bm25.get_scores(query.lower().split())
        bm25_idx = np.argsort(bm25_scores)[::-1][:candidate_pool]
        bm25_ids = df.iloc[bm25_idx]['isbn13'].tolist()

        q_emb = bi_encoder.encode([BGE_PREFIX + query], normalize_embeddings=True)
        dense_res = collection.query(
            query_embeddings=q_emb.tolist(),
            n_results=candidate_pool,
            include=['metadatas', 'documents'],
        )
        dense_ids = dense_res['ids'][0]

        # RRF
        rrf_scores = defaultdict(float)
        for rank, doc_id in enumerate(bm25_ids):
            rrf_scores[doc_id] += 1 / (60 + rank + 1)
        for rank, doc_id in enumerate(dense_ids):
            rrf_scores[doc_id] += 1 / (60 + rank + 1)

        top_ids = [d for d, _ in sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)][:candidate_pool]
        candidates = df[df['isbn13'].isin(set(top_ids))].copy()
    else:
        raise ValueError(f'Unknown first_stage: {first_stage}')

    # Stage 2 — cross-encoder rerank
    return rerank_with_cross_encoder(query, candidates, top_k=top_k)


# Smoke test
t0 = time.time()
test_res = search_with_reranking(
    'a heartbreaking story about family secrets in rural America',
    top_k=5, candidate_pool=20,
)
print(f'Reranking smoke test ({time.time()-t0:.1f}s):')
test_res[['title', 'categories', 'rerank_score']]

## 4. Candidate Pool Size Ablation

Larger pool = more recall in stage 1 = better chance of finding relevant docs.  
But larger pool = more cross-encoder calls = slower.

In [ ]:
def is_relevant(book_category, relevant_categories):
    if not isinstance(book_category, str):
        return False
    bc = book_category.lower()
    return any(rc.lower() in bc or bc in rc.lower() for rc in relevant_categories)

def precision_at_k(results_df, relevant_cats, k):
    hits = results_df.head(k)['categories'].apply(
        lambda c: is_relevant(c, relevant_cats)
    ).sum()
    return hits / k

def mrr(results_df, relevant_cats, max_k=10):
    for i, row in results_df.head(max_k).iterrows():
        if is_relevant(row['categories'], relevant_cats):
            return 1.0 / (i + 1)
    return 0.0


POOL_SIZES = [5, 10, 20, 30, 50]
SAMPLE_QUERIES = test_queries[:15]  # 15 queries for speed

pool_results = []
for pool in POOL_SIZES:
    p5_list, mrr_list, latency_list = [], [], []
    for q in SAMPLE_QUERIES:
        t0 = time.time()
        res = search_with_reranking(q['query'], top_k=5, candidate_pool=pool)
        latency_list.append((time.time() - t0) * 1000)
        p5_list.append(precision_at_k(res, q['relevant_categories'], 5))
        mrr_list.append(mrr(res, q['relevant_categories']))

    pool_results.append({
        'pool_size'   : pool,
        'P@5'         : round(np.mean(p5_list), 4),
        'MRR'         : round(np.mean(mrr_list), 4),
        'latency_ms'  : round(np.mean(latency_list), 1),
    })
    print(f'pool={pool:3d} → P@5={pool_results[-1]["P@5"]:.4f}  MRR={pool_results[-1]["MRR"]:.4f}  {pool_results[-1]["latency_ms"]:.0f}ms')

df_pool = pd.DataFrame(pool_results)

## 5. Full Evaluation — 50 Queries

In [ ]:
FINAL_POOL = 20   # chosen from ablation above

print(f'Running full evaluation (50 queries, pool={FINAL_POOL})...')
t0 = time.time()

rerank_p5, rerank_p10, rerank_mrr, rerank_latency = [], [], [], []
rerank_records = []

for q in test_queries:
    t_q = time.time()
    res = search_with_reranking(q['query'], top_k=10, candidate_pool=FINAL_POOL)
    lat = (time.time() - t_q) * 1000

    p5  = precision_at_k(res, q['relevant_categories'], 5)
    p10 = precision_at_k(res, q['relevant_categories'], 10)
    m   = mrr(res, q['relevant_categories'])

    rerank_p5.append(p5)
    rerank_p10.append(p10)
    rerank_mrr.append(m)
    rerank_latency.append(lat)
    rerank_records.append({
        'query'      : q['query'],
        'genres'     : ', '.join(q['relevant_categories']),
        'P@5'        : round(p5, 4),
        'P@10'       : round(p10, 4),
        'MRR'        : round(m, 4),
        'latency_ms' : round(lat, 1),
    })

rerank_eval = {
    'P@5'        : round(np.mean(rerank_p5),  4),
    'P@10'       : round(np.mean(rerank_p10), 4),
    'MRR'        : round(np.mean(rerank_mrr), 4),
    'latency_ms' : round(np.mean(rerank_latency), 1),
}
print(f'Done in {time.time()-t0:.1f}s')
print(f'Reranking results: {rerank_eval}')

## 6. Update Evaluation Report

In [ ]:
report_path = REPORT_PATH / 'evaluation_final.json'

if report_path.exists():
    with open(report_path) as f:
        report = json.load(f)
else:
    report = {'results': {}}

report['results']['Reranking'] = {
    **rerank_eval,
    'type'             : 'dense + cross-encoder rerank',
    'first_stage'      : 'semantic',
    'candidate_pool'   : FINAL_POOL,
    'cross_encoder'    : CROSS_ENCODER_MODEL,
    'per_query'        : rerank_records,
}

with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)
print(f'Updated: {report_path}')

## 7. Plot A — Pool Size Ablation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Quality vs pool size
ax = axes[0]
ax.plot(df_pool['pool_size'], df_pool['P@5'], marker='o', linewidth=2,
        color=MODEL_COLORS['Reranking'], label='P@5')
ax.plot(df_pool['pool_size'], df_pool['MRR'], marker='s', linewidth=2,
        color=MODEL_COLORS['Semantic'], label='MRR', linestyle='--')
ax.axvline(FINAL_POOL, color='grey', linestyle=':', linewidth=1.5,
           label=f'Chosen pool={FINAL_POOL}')
ax.set_xlabel('Candidate Pool Size (Stage 1)')
ax.set_ylabel('Score')
ax.set_title('Quality vs Pool Size', fontweight='bold')
ax.legend()

# Latency vs pool size
ax2 = axes[1]
ax2.bar(df_pool['pool_size'].astype(str), df_pool['latency_ms'],
        color=MODEL_COLORS['Reranking'], alpha=0.8, edgecolor='white')
for i, row in df_pool.iterrows():
    ax2.text(i, row['latency_ms'] + 2, f"{row['latency_ms']:.0f}ms",
             ha='center', fontsize=9)
ax2.set_xlabel('Candidate Pool Size')
ax2.set_ylabel('Latency (ms)')
ax2.set_title('Latency vs Pool Size', fontweight='bold')

plt.suptitle('Cross-encoder Reranking: Pool Size Ablation', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURE_PATH / 'eval_reranking_analysis.png', bbox_inches='tight')
plt.show()
print('Saved: eval_reranking_analysis.png')

## 8. Plot B — Final Latency vs Quality Scatter (All 5 Models)

Đây là **biểu đồ quan trọng nhất** cho báo cáo — cho thấy rõ trade-off giữa tốc độ và chất lượng.

In [ ]:
# Load all scores from final report
with open(report_path) as f:
    report = json.load(f)

# Expected latency benchmarks (ms/query on CPU)
LATENCY_ESTIMATES = {
    'TF-IDF'   : 5,
    'BM25'     : 10,
    'Semantic' : 50,
    'Hybrid'   : 60,
    'Reranking': rerank_eval['latency_ms'],
}

scatter_data = []
for model_name, model_data in report.get('results', {}).items():
    if isinstance(model_data, dict) and 'P@5' in model_data:
        scatter_data.append({
            'model'      : model_name,
            'P@5'        : float(model_data['P@5']) if model_data['P@5'] != '?' else None,
            'MRR'        : float(model_data.get('MRR', 0)) if model_data.get('MRR', '?') != '?' else None,
            'latency_ms' : LATENCY_ESTIMATES.get(model_name, model_data.get('latency_ms', 50)),
        })

df_scatter = pd.DataFrame(scatter_data).dropna(subset=['P@5'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric in zip(axes, ['P@5', 'MRR']):
    for _, row in df_scatter.iterrows():
        model = row['model']
        color = MODEL_COLORS.get(model, '#888')
        ax.scatter(
            row['latency_ms'], row[metric],
            s=200, color=color, zorder=5,
            edgecolors='white', linewidth=1.5,
        )
        ax.annotate(
            model,
            xy=(row['latency_ms'], row[metric]),
            xytext=(8, 4), textcoords='offset points',
            fontsize=9, fontweight='bold', color=color,
        )

    # "Ideal" region annotation
    ax.annotate(
        '← Faster\n↑ Better',
        xy=(0.05, 0.95), xycoords='axes fraction',
        fontsize=9, color='grey',
        ha='left', va='top',
    )

    ax.set_xlabel('Latency (ms/query)')
    ax.set_ylabel(metric)
    ax.set_title(f'Latency vs {metric} — All 5 Models', fontweight='bold')
    ax.set_xscale('log')  # log scale vì range rộng (5ms → 200ms)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0f}ms'))

# Add legend patches
import matplotlib.patches as mpatches
patches = [mpatches.Patch(color=MODEL_COLORS[m], label=m) for m in MODEL_COLORS]
fig.legend(handles=patches, loc='lower center', ncol=5, bbox_to_anchor=(0.5, -0.08))

plt.suptitle('Quality–Latency Tradeoff: Sparse vs Dense vs Hybrid vs Reranking',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURE_PATH / 'eval_latency_quality.png', bbox_inches='tight')
plt.show()
print('Saved: eval_latency_quality.png')

## 9. Final Summary Table — All 5 Models

In [ ]:
MODEL_ORDER = ['TF-IDF', 'BM25', 'Semantic', 'Hybrid', 'Reranking']
MODEL_TYPES = {
    'TF-IDF'   : 'Sparse baseline',
    'BM25'     : 'Sparse baseline',
    'Semantic' : 'Dense',
    'Hybrid'   : 'Dense + Sparse (RRF)',
    'Reranking': 'Dense + Rerank',
}

rows = []
for m in MODEL_ORDER:
    data = report['results'].get(m, {})
    rows.append({
        'Model'     : ('✓ ' if m in ['Hybrid', 'Reranking'] else '  ') + m,
        'Type'      : MODEL_TYPES.get(m, ''),
        'P@5'       : data.get('P@5', '?'),
        'P@10'      : data.get('P@10', '?'),
        'MRR'       : data.get('MRR', '?'),
        'ms/query'  : LATENCY_ESTIMATES.get(m, '?'),
    })

df_final = pd.DataFrame(rows)
print('=' * 72)
print('  FINAL RETRIEVAL COMPARISON — Precision@K + MRR (50 test queries)')
print('=' * 72)
print(df_final.to_string(index=False))
print('=' * 72)
print('  ✓ = nhóm đề xuất (proposed methods)')

## 10. Qualitative Analysis — Reranking Effect

In [ ]:
DEMO_QUERY = 'a story where someone slowly loses their grip on reality'

print(f'Query: "{DEMO_QUERY}"\n')

print('--- Stage 1: BGE-small (before reranking) ---')
stage1 = search_semantic(DEMO_QUERY, top_k=10)
print(stage1[['title', 'categories', 'score']].head(5).to_string(index=False))

print('\n--- Stage 2: After cross-encoder reranking ---')
stage2 = search_with_reranking(DEMO_QUERY, top_k=5, candidate_pool=10)
print(stage2[['title', 'categories', 'rerank_score']].to_string(index=False))

print('\n--- Rank changes ---')
stage1_ranks = {row['title']: i+1 for i, row in stage1.head(10).iterrows()}
for i, row in stage2.iterrows():
    old_rank = stage1_ranks.get(row['title'], '?')
    new_rank = i + 1
    direction = '↑' if isinstance(old_rank, int) and old_rank > new_rank else ('↓' if isinstance(old_rank, int) and old_rank < new_rank else '=')
    print(f'  {direction} {row["title"][:45]:45s}  {old_rank} → {new_rank}')

---
## Done ✓ — Tất cả 9 notebooks hoàn thành

**Artifacts:**
- `reports/evaluation_final.json` — đầy đủ 5 models
- `reports/figures/eval_reranking_analysis.png`
- `reports/figures/eval_latency_quality.png` ← **dùng trong slide**

**Thứ tự chạy bắt buộc:**
```
01 → 02 → 03 → 04 → 05 → 06 → 07 → 08 → 09
```

**Next:** Backend FastAPI (`backend/main.py`) + Frontend React.